In [1]:
import pandas as pd
import sys
sys.path.append("..")

from src.config import GOLD_DIR

pd.set_option("display.float_format", "{:,.2f}".format)

fct_sales = pd.read_parquet(GOLD_DIR / "fct_sales.parquet")
dim_products = pd.read_parquet(GOLD_DIR / "dim_products.parquet")
dim_customers = pd.read_parquet(GOLD_DIR / "dim_customers.parquet")

In [2]:
fct_sales.head(3)

,sale_id,product_id,customer_id,quantity,total_amount,sale_date,has_product,has_customer,amount_is_valid,category,price,region,signup_date,after_signup,unit_price
0,5083,343,9320,9,1431,2023-06-19,True,True,True,doll,159.00,NaN,2021-11-08,True,159.00
1,6690,463,8458,6,648,2020-05-15,True,True,True,rc toy,108.00,South,2022-09-01,False,108.00
2,8362,750,8226,1,172,2022-11-03,True,True,True,action figure,172.00,West,2023-10-16,False,172.00


In [3]:
dim_products.head(3)

,product_id,product_name,category,price,supplier
0,207,NaN,puzzle,64,Brown-Garcia
1,732,Joseph,rc toy,170,Ortiz Inc
2,463,Jacob,rc toy,108,Hinton-Patterson


In [4]:
dim_customers.head(3)

,customer_id,name,region,signup_date,email,loyalty_points
0,8220,Brandon Bender,North,2023-08-14,mirandajoel@example.org,129
1,6670,Kari Ford,North,2020-03-02,brucewesley@example.org,528
2,9865,Andrea Leonard,NaN,2020-11-26,newmantimothy@example.com,2182


In [5]:
# The brief asks for the latest 6 months in the dataset, so windows are anchored to max(sale_date)
ANCHOR = fct_sales["sale_date"].max()

In [6]:
def sales_window(fct, months, anchor=ANCHOR):
    # Rolling window of the last N months, exclusive at the start, inclusive at the end.
    start = anchor - pd.DateOffset(months=months)
    return fct[(fct["sale_date"] > start) & (fct["sale_date"] <= anchor)]


def total_sales_by_category(fct, months=6, anchor=ANCHOR):
    """
    Total sales per product category over the latest N months in the dataset.

    Returns revenue, units and transaction count per category, revenue-sorted.
    Comment: 
     - Rows with a corrupt amount are excluded. 
     - Rows with no category are kept and reported as 'Unknown' so the totals still reconcile.
    """
    window = sales_window(fct, months, anchor)
    valid = window[window["amount_is_valid"]].copy()
    valid["category"] = valid["category"].fillna("Unknown")

    return (valid
            .groupby("category")
            .agg(total_sales=("total_amount", "sum"),
                 units=("quantity", "sum"),
                 transactions=("sale_id", "count"))
            .sort_values("total_sales", ascending=False))

## Total sales per category, latest 6 months

In [7]:
start = ANCHOR - pd.DateOffset(months=6)
print(f"Sales per category between: {start.date()} -> {ANCHOR.date()}")

result_6m = total_sales_by_category(fct_sales, months=6)
result_6m

Sales per category between: 2023-02-09 -> 2023-08-09


,total_sales,units,transactions
category,,,
puzzle,61879,532,97
action figure,60817,511,90
rc toy,50886,424,71
doll,49581,407,67
board game,32336,296,52
Unknown,6198,41,8


## Q1 - Which product category has the highest sales volume in the **North** region over the past 3 months?

**Assumption:**

 - **"Volume" is ambiguous** between units and revenue, so both are reported (but I will assume the desired answer should target units).

**Personal note:**
- Region is a customer attribute, so "sales in the North region" is read as "sales made by customers registered in North".


In [8]:
def sales_by_category_in_region(fct, region, months=3, anchor=ANCHOR):
    window = sales_window(fct, months, anchor)
    df = window[(window["region"] == region) & window["amount_is_valid"]]

    return (df
            .groupby("category")
            .agg(units=("quantity", "sum"),
                 revenue=("total_amount", "sum"),
                 transactions=("sale_id", "count"))
            .sort_values("units", ascending=False))


start_3m = ANCHOR - pd.DateOffset(months=3)
print(f"North metrics per category, between: {start_3m.date()} -> {ANCHOR.date()}")

q1 = sales_by_category_in_region(fct_sales, "North", months=3)
q1

North metrics per category, between: 2023-05-09 -> 2023-08-09


,units,revenue,transactions
category,,,
puzzle,82,10507,15
doll,79,9083,10
rc toy,65,6463,10
action figure,46,5183,8
board game,25,2277,4


### Answer Q1:
- "Puzzle" has the highest sales volume in the **North** region over the past 3 months, leading with 82 units (very close to doll, which has 79 units sold... this is very close, so its important to note that probably with a few additional days of data, the answer could change).
- There is a caveat (details below), 88/400 ~22% of customers and ~23% of sales, does not have a region assigned. So results may drift from reality. This problem comes from the source "customers.json", so this source should be checked.

In [9]:
# Sanity check of Region column
#dim_customers['region'].value_counts(dropna=False)
print(f"customers with no region:  {dim_customers['region'].isna().sum()} of {len(dim_customers)}")
print(f"fct_sales with no region:  {fct_sales['region'].isna().sum()} of {len(fct_sales)}")

customers with no region:  88 of 400
fct_sales with no region:  706 of 2983


## Q2: Identify the **top 5 customers** by sales value since their signup date.

**Thinking process:** First we have to create a "cohort", made out of sales and customers that satisfy (at least) the condition: 
- Product date sold to customer >= customer signup date 

*PD: As an assumption, I put another condition: Sale's amount should be positive... this should be pre-defined with stakeholders beforehand.*

In [10]:
def top_customers_since_signup(fct, dim_customers, n=5):
    # We check that their sales are after signup and that the amount is a valid number.
    eligible = fct[fct["after_signup"] & fct["amount_is_valid"]]

    return (eligible
            .groupby("customer_id")
            .agg(sales_value=("total_amount", "sum"),
                 transactions=("sale_id", "count"))
            .merge(dim_customers[["customer_id", "name", "region", "signup_date"]],
                   on="customer_id", how="left")
            .sort_values("sales_value", ascending=False)
            .head(n)
            .reset_index(drop=True))


q2 = top_customers_since_signup(fct_sales, dim_customers)
q2

,customer_id,sales_value,transactions,name,region,signup_date
0,1676,10204,16,Craig Pearson,NaN,2020-02-21
1,1991,9635,10,Stephen Rivera,North,2019-10-05
2,3116,9389,12,Zachary Rodgers,South,2019-03-09
3,9510,8825,12,Michelle Taylor,NaN,2020-11-22
4,5263,7837,12,Kathleen Lyons MD,West,2019-09-22


### Answer Q2:
The top 5 customers by sales value **since their signup date** are: 
- Craig Pearson
- Stephen Rivera
- Zachary Rodgers
- Michelle Taylor
- Kathleen Lyons MD

*PD: As an assumption, I put another condition: Sale's amount should be positive... this should be pre-defined with stakeholders beforehand.*

In [11]:
# Minor sanity checks
valid = fct_sales[fct_sales["amount_is_valid"]]
has_signup = fct_sales["signup_date"].notna()

print(f"sales with valid amount: {len(valid)}")
print(f"of which after signup: {(valid['after_signup']).sum()}")
print(f"before signup: {(has_signup & ~fct_sales['after_signup'] & fct_sales['amount_is_valid']).sum()}")
print(f"customer has no signup: {(~has_signup & fct_sales['amount_is_valid']).sum()}")

sales with valid amount: 2931
of which after signup: 1211
before signup: 1240
customer has no signup: 480


**Comment:** The filter discards ~42% of the data, this could be important to know for a stakeholder.

## Q3: Are there any products that have **never been sold**? If yes, list them.

**Thinking process**: This requires an "anti-join". First we have to define a set including all product id's that have been sold, then we check in our products if there are any ids that are not in the defined set (implying they have never been sold). 

In [12]:
def products_never_sold(fct, dim_products):
    sold = set(fct["product_id"].dropna())
    return dim_products[~dim_products["product_id"].isin(sold)]


q3 = products_never_sold(fct_sales, dim_products)

print(f"products in catalogue:  {len(dim_products)}")
print(f"distinct products sold: {fct_sales['product_id'].nunique()}")
print(f"never sold:             {len(q3)}")

q3

products in catalogue:  200
distinct products sold: 200
never sold:             0


,product_id,product_name,category,price,supplier


### Answer Q3:
There are no products that have never been sold.

In [13]:
# Sanity checks: Sales that reference no valid product at all, this is a separate problem, but its something that could clutter our insights.
null_prod = fct_sales["product_id"].isna()
rows, total_rows = null_prod.sum(), len(fct_sales)
rev, total_rev = fct_sales.loc[null_prod, "total_amount"].sum(), fct_sales["total_amount"].sum()

print(f"sales with null product_id: {rows} of {total_rows} ({rows/total_rows:.1%})")
print(f"revenue affected: {rev:,.0f} of {total_rev:,.0f} ({rev/total_rev:.1%})")

# ~3% of the sales don't have a valid id. 
# This should be reported but keeping in mind that is too small to shift any conclusion (given the impact it would not be a priority).

sales with null product_id: 87 of 2983 (2.9%)
revenue affected: 46,456 of 1,758,824 (2.6%)


## Q4: Calculate the **average sales price** of the `Action Figure` category products.

**Thinking process**: We add as a condition that category has to be equal to `Action Figure` (In my case I standardize the column so now it is `action figure`, this could be documented as a column description in a data mart)

In [14]:
def average_sales_price(fct, dim_products, category="action figure"):
    catalogue = dim_products[dim_products["category"] == category]
    sales = fct[(fct["category"] == category) & fct["amount_is_valid"]]

    return pd.Series({
        "products_in_catalogue": len(catalogue),
        "avg_catalogue_price": catalogue["price"].mean(),
        "transactions": len(sales),
        "avg_transaction_price": sales["unit_price"].mean(),
        "weighted_avg_price": sales["total_amount"].sum() / sales["quantity"].sum(),
    })


q4 = average_sales_price(fct_sales, dim_products)
q4

products_in_catalogue    51.00
avg_catalogue_price     111.31
transactions            727.00
avg_transaction_price   112.81
weighted_avg_price      113.72
dtype: float64

### Answer Q4:

Details above, the **average sales price** of the `Action Figure` category is **$112.81**

Additional metrics:
- products_in_catalogue    51.00
- avg_catalogue_price     111.31
- transactions            727.00
- **avg_transaction_price   112.81**
- weighted_avg_price      113.72

In [15]:
# Sanity check: To showcase that there were inconsistent formattings in the category of Action Figure (appearing also as Action-Figure)

from src.config import BRONZE_DIR

raw_categories = pd.read_parquet(BRONZE_DIR / "products.parquet")["Category"].value_counts()
print("Raw categories in products.csv:")
print(raw_categories.to_string())

print()

print("After cleaning:")
print(dim_products["category"].value_counts().to_string())

Raw categories in products.csv:
Category
Puzzle           45
RC Toy           43
Action Figure    37
Doll             35
Board Game       26
Action-Figure    15

After cleaning:
category
action figure    51
puzzle           45
rc toy           43
doll             35
board game       26


## Summary

| # | Question | Answer |
|---|---|---|
| - | Total sales by category, latest 6 months | See table above |
| 1 | Highest sales volume, North, past 3 months | "Puzzle" has the highest sales volume in the **North** region over the past 3 months |
| 2 | Top 5 customers by value since signup | (Craig Pearson, Stephen Rivera, Zachary Rodgers, Michelle Taylor, Kathleen Lyons MD) |
| 3 | Products never sold | None, all catalogue products appear in sales |
| 4 | Average sales price of Action Figure | $112.81, see table above for more detail |

**Comments:** 
- Every figure above excludes the 52 sales with a negative `total_amount` (it was an assumption, maybe in real life there could be cases where it makes sense to keep them, but in order to figure that we would need to ask other teams). 
- Regional and per-customer figures exclude sales that cannot be joined to a customer record.